Syntax:

- Time (t)
- Action (A)
- State (S)
- Reward (R)

RL Basics Recap

- Utility Function: Any function that is applying transformations (updates) to the agent's action or state and returns a value that supports the agent's knowledge and decision making for its future steps. E.g. value functions, reward functions.


#7.1: MDP

**Example 7.1: Aircraft Collision Problem as an MDP**

- **S**: State
- **A**: Action 
- **T**: Transition function
- **R**: Reward function
- **TR**: Sample Transition & Reward function 
- **y**: Discount (0-1) that decides how much you value the future

Given current state s_t:
  1) choose action a_t ~ π(a | s_t)
  2) environment returns reward r_t = R(s_t, a_t)
  3) environment transitions to next state s_k ~ P(s' | s_t, a_t)
    (here s_k ≡ s_{t+1})

Loop/Update:
  s_t → a_t → (r_t, s_{t+1}) → repeat

In [8]:
class MDP:
    def __init__(self, n_states: int, n_actions: int, _reward: dict):
        self.n_states = n_states
        self.n_actions = n_actions

        self.s = None
        self.a = None
        self._reward = _reward

    def step(self):
        # transition function
        pass

    def reward(self):
        # reward function
        pass

    def transition_reward(self):
        # transition & reward function
        pass

In [ ]:
# Tiny 3-state MDP: (S, A, T, R, γ) + one-step simulation showing Markov property

import random

# --- (S, A, T, R, γ) ---
S = ["Safe", "Near", "Collision"]              # states
A = ["Hold", "Climb"]                          # actions
gamma = 0.95

# T: transition probabilities T[s][a] -> list of (s_next, prob) NOTE: unseen to the agent
T = {
    "Safe": {
        "Hold":  [("Safe", 0.8), ("Near", 0.2)],
        "Climb": [("Safe", 0.9), ("Near", 0.1)],
    },
    "Near": {
        "Hold":  [("Near", 0.6), ("Collision", 0.4)],
        "Climb": [("Safe", 0.7), ("Near", 0.25), ("Collision", 0.05)],
    },
    "Collision": {
        "Hold":  [("Collision", 1.0)],
        "Climb": [("Collision", 1.0)],
    },
}

# R: reward function R(s, a, s')
def R(s, a, s_next):
    if s_next == "Collision": return -100.0
    if s_next == "Safe":      return +1.0
    return -1.0  # "Near"

def sample_next(s, a):
    roll = random.random()
    cum = 0.4
    for s_next, p in T[s][a]:
        cum += p
        if roll <= cum:
            return s_next
    return T[s][a][-1][0]  # fallback

# --- One step: s_t -> a_t -> (r_t, s_{t+1}) ---
s_t = "Near"
a_t = "Hold"
s_tp1 = sample_next(s_t, a_t)
r_t = R(s_t, a_t, s_tp1)

print("One-step transition:")
print(f"  s_t={s_t}, a_t={a_t}  ->  s_(t+1)={s_tp1}, r_t={r_t}, γ={gamma}")

# --- Markov assumption demo: history doesn't matter once s_t and a_t are fixed ---
fake_histories = [
    ["Safe", "Safe", "Near"],          # history 1 ends at s_t="Near"
    ["Collision", "Near"],             # history 2 ends at s_t="Near"
    ["Safe", "Near", "Near", "Near"],  # history 3 ends at s_t="Near"
]
print("\nMarkov check (same s_t and a_t => same T distribution, regardless of history):")
for h in fake_histories:
    print(f"  history={h}  =>  P(. | s_t='Near', a_t='Hold') = {T['Near']['Hold']}")

One-step transition:
  s_t=Near, a_t=Hold  ->  s_(t+1)=Near, r_t=-1.0, γ=0.95

Markov check (same s_t and a_t => same T distribution, regardless of history):
  history=['Safe', 'Safe', 'Near']  =>  P(. | s_t='Near', a_t='Hold') = [('Near', 0.6), ('Collision', 0.4)]
  history=['Collision', 'Near']  =>  P(. | s_t='Near', a_t='Hold') = [('Near', 0.6), ('Collision', 0.4)]
  history=['Safe', 'Near', 'Near', 'Near']  =>  P(. | s_t='Near', a_t='Hold') = [('Near', 0.6), ('Collision', 0.4)]


In [4]:
sample_next(s_t, a_t)

'Near'

In [12]:
from dataclasses import dataclass

@dataclass
class Agent:
    name: str
    gamma: float  # discount factor

    def act(self, delay_steps: int, r_now: float, r_later: float) -> str:
        # Compare discounted value of waiting vs taking now
        v_take = r_now
        v_wait = (self.gamma ** delay_steps) * r_later
        return "wait" if v_wait > v_take else "take"

def simulate(agent: Agent, horizon: int = 30, delay_steps: int = 2,
             r_now: float = 1.0, r_later: float = 3.0) -> float:
    # Rewards can arrive later; we track a small "pending rewards" queue by time index.
    pending = [0.0] * (horizon + delay_steps + 1)
    total = 0.0

    for t in range(horizon):
        # collect any reward arriving at this step
        total += pending[t]

        action = agent.act(delay_steps, r_now, r_later)
        if action == "take":
            total += r_now
        else:
            pending[t + delay_steps] += r_later

    return total

if __name__ == "__main__":
    env = dict(horizon=30, delay_steps=2, r_now=1.0, r_later=3.0)

    impulsive = Agent("Impulsive", gamma=0.4)
    patient   = Agent("Patient",   gamma=0.95)

    for a in (impulsive, patient):
        score = simulate(a, **env)
        print(f"{a.name:10s} (γ={a.gamma}) total reward: {score:.1f}")

    # Quick sanity check of the decision rule:
    # take if 1 > (gamma^2)*3  => gamma < sqrt(1/3) ≈ 0.577

Impulsive  (γ=0.4) total reward: 30.0
Patient    (γ=0.95) total reward: 84.0
